Data source: https://www.kaggle.com/competitions/iaa-ai-assisted-hackathon-2026-part-1

In [0]:
# Seaborn and openpyxl (needed for Excel exports) are not installed in this environment
%pip install seaborn openpyxl

In [0]:
# Import libraries:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

#### Loading Data

In [0]:
import os

# Load every .parquet file in the data folder of the current branch
data_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/data"
parquet_files = sorted(f for f in os.listdir(data_dir) if f.lower().endswith(".parquet"))

dataframes = {}   # dataset name -> Spark DataFrame
row_counts = {}   # dataset name -> total row count

for file_name in parquet_files:
    name = file_name.replace(".parquet", "")
    df = spark.read.parquet(os.path.join(data_dir, file_name))
    dataframes[name] = df
    globals()[f"df_{name}"] = df   # each dataset is also its own variable, e.g. df_claims

# Report how many files were imported and their names
print(f"Imported {len(dataframes)} parquet files from {data_dir}:\n")
for name, df in dataframes.items():
    row_counts[name] = df.count()
    print(f"  df_{name:<18} | {row_counts[name]:>10,} rows | {len(df.columns)} columns")

In [0]:
import pandas as pd

# Folder for all Excel exports (created if it doesn't exist)
excel_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files"
os.makedirs(excel_dir, exist_ok=True)

def _sheet_name(name):
    r"""Excel sheet names: max 31 chars, cannot contain : \ / ? * [ ]"""
    return "".join(c for c in name if c not in ":\\/?*[]")[:31] or "sheet"

overview_path = os.path.join(excel_dir, "Data_overview_100_rows.xlsx")

with pd.ExcelWriter(overview_path, engine="openpyxl") as writer:
    for name, df in dataframes.items():
        sheet = _sheet_name(name)
        # Line at the top of the sheet stating the total number of rows
        summary = pd.DataFrame([[f"{name}: {row_counts[name]:,} total rows (showing first 100)"]])
        summary.to_excel(writer, sheet_name=sheet, index=False, header=False, startrow=0)
        # First 100 rows of the dataframe, a few rows below the summary line
        df.limit(100).toPandas().to_excel(writer, sheet_name=sheet, index=False, startrow=3)

print(f"Overview workbook saved to: {overview_path}")
print(f"Contains one sheet per dataset: {', '.join(dataframes.keys())}")

In [0]:
# Export only the claims dataframe in full to Excel.
# The other datasets are too large to be practical in Excel — use the
# Data_overview_100_rows.xlsx workbook to preview them instead.
claims_path = os.path.join(excel_dir, "claims.xlsx")
dataframes["claims"].toPandas().to_excel(claims_path, engine="openpyxl", index=False)
print(f"Saved claims.xlsx ({row_counts['claims']:,} rows) to: {excel_dir}")

In [0]:
from pyspark.sql import functions as F

# Build a summary workbook: an Overview sheet (one row per dataset) plus a sheet
# per dataset with column dtypes, missing values and describe statistics.
summary_path = os.path.join(excel_dir, "Data_Summary.xlsx")

overview_rows = []
per_dataset_stats = {}   # dataset name -> column-level stats DataFrame
shapes = {}              # dataset name -> (rows, columns)

for name, df in dataframes.items():
    n_rows = df.count()
    dtypes = dict(df.dtypes)
    cols = list(dtypes.keys())
    numeric_cols = [c for c, t in dtypes.items()
                    if t in ("int", "bigint", "smallint", "tinyint", "double", "float")
                    or t.startswith("decimal")]
    shapes[name] = (n_rows, len(cols))

    # Missing values per column (single aggregation pass)
    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first()
    null_counts = {c: null_row[c] for c in cols}

    stats = pd.DataFrame({
        "column": cols,
        "dtype": [dtypes[c] for c in cols],
        "missing_values": [null_counts[c] for c in cols],
        "missing_%": [round(100.0 * null_counts[c] / n_rows, 2) if n_rows else 0.0 for c in cols],
    })

    # Describe statistics (count/mean/stddev/min/max) for numeric columns
    if numeric_cols:
        desc = (df.select(numeric_cols).describe()
                .toPandas().set_index("summary").T
                .apply(pd.to_numeric, errors="coerce"))
        stats = stats.merge(desc, left_on="column", right_index=True, how="left")
    per_dataset_stats[name] = stats

    total_missing = sum(null_counts.values())
    overview_rows.append({
        "dataframe": name,
        "rows": n_rows,
        "columns": len(cols),
        "numeric_columns": len(numeric_cols),
        "total_missing_values": total_missing,
        "missing_%_of_cells": round(100.0 * total_missing / (n_rows * len(cols)), 2) if n_rows and cols else 0.0,
    })

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
    overview = pd.DataFrame(overview_rows)
    overview.to_excel(writer, sheet_name="Overview", index=False)
    for name, stats in per_dataset_stats.items():
        sheet = _sheet_name(name)
        n_rows, n_cols = shapes[name]
        header = pd.DataFrame([[f"{name}: {n_rows:,} rows x {n_cols} columns"]])
        header.to_excel(writer, sheet_name=sheet, index=False, header=False, startrow=0)
        stats.to_excel(writer, sheet_name=sheet, index=False, startrow=2)

print(f"Summary workbook saved to: {summary_path}")
display(overview)

#### Create master policy files

In [0]:
# Loop through motor/contents/building to create 3 master policy files with 
# informtaion from the other files appended.
# Note that not all information will be carried through - e.g. if there are multiple
# claims for a policy row we can only bring through one line - if this happens we will 
# note if there are multiple perils and the total count of claims for that row
#
# NOTE: the addresses file has no customer_id column, so addresses is joined on
# address_id (carried on the policy files) instead.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Claim fields to take from the first claim (lowest claim_id) for each policy
CLAIM_FIRST_COLS = ["claim_id", "loss_date", "report_date", "status", "repairer_id", "third_party_id"]

# Weather join keys renamed so they don't clash with the policy file's own columns
wx = (dataframes["weather"]
      .withColumnRenamed("address_id", "_wx_address_id")
      .withColumnRenamed("year", "_wx_year"))

master_dataframes = {}   # e.g. {"policies_motor_master": DataFrame}

for line_name, pol_name in [("motor", "policies_motor"),
                            ("contents", "policies_contents"),
                            ("buildings", "policies_buildings")]:
    pol = dataframes[pol_name]

    # HUMAN DECISION (2026-10-20, may be revisited): keep only policies whose
    # renewal_decision is 'renewed' (~81-84% of rows per line). This drops
    # lapsed_by_customer / cancelled / declined_by_insurer / mid_term_change rows,
    # including short-period cancellations such as single-day policies.
    pol = pol.filter(F.col("renewal_decision") == "renewed")

    # ---- Claims: keep only this line, aggregate per policy_id and loss year ----
    # (aggregated before joining so the join can never fan out policy rows)
    cl = (dataframes["claims"]
          .filter(F.col("line") == line_name)
          .withColumn("_claim_year", F.year("loss_date"))
          .withColumn("_rn", F.row_number().over(
              Window.partitionBy("policy_id", "_claim_year").orderBy("claim_id"))))

    cl_agg = (cl.groupBy("policy_id", "_claim_year")
              .agg(
                  F.sum("paid_to_date").alias("paid_to_date"),
                  F.sum("incurred_to_date").alias("incurred_to_date"),
                  F.sum("recoveries").alias("recoveries"),
                  F.count(F.lit(1)).alias("n_claims"),
                  F.collect_set("peril").alias("_perils"),
                  *[F.max(F.when(F.col("_rn") == 1, F.col(c))).alias(c) for c in CLAIM_FIRST_COLS])
              .withColumn("peril",
                          F.when(F.size(F.col("_perils")) > 1, F.lit("multiple"))
                           .otherwise(F.element_at(F.col("_perils"), 1)))
              .drop("_perils")
              .withColumnRenamed("policy_id", "_cl_policy_id"))

    # ---- Master file: policies + weather + customers + addresses + claims ----
    master = (
        pol
        # weather at the policy's address for the policy year
        .join(wx,
              (F.col("address_id") == F.col("_wx_address_id")) &
              (F.year(F.col("period_start")) == F.col("_wx_year")),
              "left")
        .drop("_wx_address_id", "_wx_year")
        # customer details
        .join(dataframes["customers"], "customer_id", "left")
        # address details - joined on address_id (addresses has no customer_id)
        .join(dataframes["addresses"], "address_id", "left")
        # claims on this line whose loss year matches the policy year
        .join(cl_agg,
              (F.col("policy_id") == F.col("_cl_policy_id")) &
              (F.year(F.col("period_start")) == F.col("_claim_year")),
              "left")
        .drop("_cl_policy_id", "_claim_year")
        # claim_indicator: 1 if the policy row has any claims (n_claims >= 1), 0 otherwise
        .withColumn("claim_indicator", F.when(F.col("n_claims") >= 1, 1).otherwise(0))
        # incurred net of recoveries: aggregated incurred_to_date minus aggregated
        # recoveries (null for policy-years with no claims)
        .withColumn("incurred_net_of_recoveries", F.col("incurred_to_date") - F.col("recoveries"))
        # ultimate_claims: for now equal to incurred_to_date - recoveries; may be
        # revised later if claims are developed to ultimate (see AI_ASSISTANCE.md)
        .withColumn("ultimate_claims", F.col("incurred_to_date") - F.col("recoveries"))
        # earned_premium: the annualized written_premium earned over the days actually
        # in force, pro-rated by the number of days in the policy's calendar year.
        # DECISION / ASSUMPTION (2026-10-20, flagged in AI_ASSISTANCE.md): written_premium
        # is treated as the ANNUAL premium under the insurer's current pricing (evidence:
        # premium is unrelated to term length, corr ~0; the same policy pays ~99.6% as much
        # for a ~104-day period as for a following full year). Day counts are inclusive
        # of both endpoints, e.g. 2021-01-01 to 2021-12-31 = 365 days (366 in 2024).
        # earned_premium = written_premium x days_in_force / days_in_calendar_year
        .withColumn("_days_total", F.datediff("period_end", "period_start") + 1)
        .withColumn("_year_end", F.make_date(F.year("period_start"), F.lit(12), F.lit(31)))
        .withColumn("_days_earned", F.least(F.col("_days_total"),              # days in force during
                                            F.datediff("_year_end", "period_start") + 1))  # this calendar year
        .withColumn("_year_days", F.datediff("_year_end",
                                             F.make_date(F.year("period_start"), F.lit(1), F.lit(1))) + 1)
        .withColumn("earned_premium",
                    F.round(F.col("written_premium") * F.col("_days_earned") / F.col("_year_days"), 2))
        .drop("_days_total", "_year_end", "_days_earned", "_year_days")
    )

    master_name = f"{pol_name}_master"
    master_dataframes[master_name] = master
    globals()[f"df_{master_name}"] = master   # e.g. df_policies_motor_master

    # Sanity checks in one pass: master row count must equal the renewed policy rows (no fan-out)
    n_pol_all = dataframes[pol_name].count()
    n_pol = pol.count()
    s = master.agg(
        F.count(F.lit(1)).alias("rows"),
        F.sum(F.when(F.col("n_claims").isNotNull(), 1).otherwise(0)).alias("with_claims"),
        F.sum("claim_indicator").alias("claim_ind_total"),
        F.sum(F.when(F.col("earned_premium").isNull(), 1).otherwise(0)).alias("earned_null"),
        F.round(F.sum("incurred_to_date") - F.sum("recoveries"), 0).alias("chk_net"),
        F.round(F.sum("ultimate_claims"), 0).alias("ult_total"),
        F.sum(F.when(F.col("earned_premium") > F.col("written_premium"), 1).otherwise(0)).alias("earned_over"),
        F.sum(F.when(F.col("hail_days").isNotNull(), 1).otherwise(0)).alias("with_weather"),
        F.sum(F.when(F.col("surname").isNotNull(), 1).otherwise(0)).alias("with_customer"),
        F.sum(F.when(F.col("region").isNotNull(), 1).otherwise(0)).alias("with_address"),
    ).first()
    print(f"{master_name}: {s['rows']:,} rows (renewed policies {n_pol:,} of {n_pol_all:,}, match: {s['rows'] == n_pol}) | "
          f"claims matched: {s['with_claims']:,} ({100.0 * s['with_claims'] / s['rows']:.1f}%) | "
          f"weather: {s['with_weather']:,} | customers: {s['with_customer']:,} | "
          f"addresses: {s['with_address']:,} | claim_indicator total: {s['claim_ind_total']:,} "
          f"(consistent with claims: {s['claim_ind_total'] == s['with_claims']}) | "
          f"earned_premium nulls: {s['earned_null']:,} | earned > written: {s['earned_over']:,} | "
          f"ultimate_claims total: {s['ult_total']:,.0f} (consistent: {s['ult_total'] == s['chk_net']})")

  

In [0]:
# Save the first 100 rows of each master policy file to its own Excel workbook
for name, master_df in master_dataframes.items():
    out_path = os.path.join(excel_dir, f"{name}_first_100_rows.xlsx")
    master_df.limit(100).toPandas().to_excel(out_path, engine="openpyxl", index=False)
    print(f"Saved {os.path.basename(out_path)} to: {excel_dir}")

In [0]:
# Examples to verify the earned_premium calculation (from the motor master file)
# earned_premium = written_premium (annual) x days in force / days in the calendar year
ex = (df_policies_motor_master
      .filter(F.col("period_start").isNotNull() & F.col("period_end").isNotNull() & F.col("written_premium").isNotNull())
      .withColumn("days_total", F.datediff("period_end", "period_start") + 1)
      .withColumn("year_days", F.datediff(
          F.make_date(F.year("period_start"), F.lit(12), F.lit(31)),
          F.make_date(F.year("period_start"), F.lit(1), F.lit(1))) + 1)
      .select("policy_id", "period_start", "period_end", "written_premium",
              "days_total", "year_days", "earned_premium"))

# Show a spread of real examples: January starts, mid-year starts, and very short periods
jan_full = ex.filter(F.month("period_start") == 1).orderBy(F.desc("days_total")).limit(2)
mid_year = ex.filter(F.month("period_start") == 7).limit(2)
short = ex.orderBy("days_total").limit(2)

print("Real motor policies (earned = written x days in force / days in year):")
display(jan_full)
display(mid_year)
display(short)

# Synthetic rows (NOT from the data) demonstrating the formula, incl. the 2024 leap year
syn = (spark.createDataFrame(
        [("TEST_FULL_2024", "2024-01-01", "2024-12-31", 500.0),   # full leap-year policy: 366/366
         ("TEST_90D_2021", "2021-07-01", "2021-09-28", 500.0)],  # 90-day policy: 90/365
        ["policy_id", "period_start", "period_end", "written_premium"])
       .withColumn("period_start", F.to_date("period_start"))
       .withColumn("period_end", F.to_date("period_end"))
       .withColumn("days_total", F.datediff("period_end", "period_start") + 1)
       .withColumn("year_days", F.datediff(
           F.make_date(F.year("period_start"), F.lit(12), F.lit(31)),
           F.make_date(F.year("period_start"), F.lit(1), F.lit(1))) + 1)
       .withColumn("earned_premium",
                   F.round(F.col("written_premium") * F.col("days_total") / F.col("year_days"), 2)))

print("Synthetic rows (not from the data):")
display(syn)
print("Hand check: TEST_FULL_2024 = 500.00 x 366/366 = 500.00 | TEST_90D_2021 = 500.00 x 90/365 = 123.29")

# Save the examples to Excel for review
ex_examples = (jan_full.unionByName(mid_year)
               .unionByName(short)
               .unionByName(syn.select("policy_id", "period_start", "period_end", "written_premium",
                                       "days_total", "year_days", "earned_premium"))).distinct()
ex_path = os.path.join(excel_dir, "earned_premium_examples.xlsx")
ex_examples.toPandas().to_excel(ex_path, engine="openpyxl", index=False)
print(f"\nSaved examples to: {ex_path}")

In [0]:
# Examples of the claims amount fields (from the motor master file)
# incurred_net_of_recoveries and ultimate_claims = incurred_to_date - recoveries
# (aggregated sums where a policy-year has multiple claims; null when no claims)
claim_cols = ["policy_id", "period_start", "n_claims", "incurred_to_date", "recoveries",
              "incurred_net_of_recoveries", "ultimate_claims"]

print("Single-claim rows:")
display(df_policies_motor_master.filter(F.col("n_claims") == 1).select(*claim_cols).limit(5))

print("Multi-claim rows (incurred/recoveries are sums across claims):")
display(df_policies_motor_master.filter(F.col("n_claims") > 1).select(*claim_cols).limit(5))

In [0]:
# Overview workbook for the three master policy files: one sheet per line
# (motor, contents, buildings), same layout as Data_overview_100_rows.xlsx
master_overview_path = os.path.join(excel_dir, "Master_Data_overview_100_rows.xlsx")

with pd.ExcelWriter(master_overview_path, engine="openpyxl") as writer:
    for name, master_df in master_dataframes.items():
        sheet = _sheet_name(name)   # e.g. policies_motor_master
        # Line at the top of the sheet stating the total number of rows
        summary = pd.DataFrame([[f"{name}: {master_df.count():,} total rows (showing first 100)"]])
        summary.to_excel(writer, sheet_name=sheet, index=False, header=False, startrow=0)
        # First 100 rows of the master dataframe, a few rows below the summary line
        master_df.limit(100).toPandas().to_excel(writer, sheet_name=sheet, index=False, startrow=3)

print(f"Master overview workbook saved to: {master_overview_path}")
print(f"Contains one sheet per master file: {', '.join(master_dataframes.keys())}")

#### Feature selection

In [0]:
# Feature selection for the motor Tweedie XGBoost — HUMAN DECISION (2026-10-20):
# expert-judgement first pass, to be revisited after EDA (see AI_ASSISTANCE.md).
# Target = ultimate_claims (nulls -> 0 for claim-free policy-years).
# 'Year' is derived from period_start. All claims-derived columns are excluded as
# features (leakage); IDs are kept only for GroupKFold grouping and prediction tracking.

MOTOR_FEATURES = [
    "Year",                                     # derived from period_start
    # exposure & premium
    "exposure", "written_premium", "earned_premium",
    # driver & vehicle
    "driver_age", "years_licensed", "ncb_years", "vehicle_value_group", "vehicle_age",
    "make_model", "engine_size_cc", "body_type", "transmission", "seats", "fuel_type",
    "driver_assist", "cover_type", "annual_mileage", "overnight_parking", "vehicle_use",
    "named_young_driver", "telematics_opt_in", "telematics_score",
    "voluntary_excess", "compulsory_excess",
    # policy admin
    "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
    "days_quote_to_inception",
    # customer
    "occupation", "marital_status", "credit_band", "prior_claims_5y", "licence_points",
    "branch_id", "marketing_segment",
    # address (motor-relevant)
    "region", "urban_density",
    # weather (hail only for motor)
    "hail_days",
]

TARGET = "ultimate_claims"
GROUP_COL = "customer_id"              # GroupKFold grouping: all years of one customer stay together
TRACK_COLS = ["policy_id", "row_id"]  # for attaching predictions; never model inputs

# Every listed column must exist in the master (Year is derived here, not in the master)
_missing = [c for c in MOTOR_FEATURES + [TARGET, GROUP_COL, *TRACK_COLS]
            if c != "Year" and c not in df_policies_motor_master.columns]
assert not _missing, f"Listed columns missing from the master: {_missing}"

motor_model = (df_policies_motor_master
               .withColumn("Year", F.year("period_start"))
               .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0)))   # no claims -> 0
               .select(*TRACK_COLS, GROUP_COL, *MOTOR_FEATURES, TARGET))

# One pass over the assembled dataset: size, target behaviour, feature missingness
s = motor_model.agg(
    F.count(F.lit(1)).alias("_rows"),
    F.sum(F.when(F.col(TARGET) > 0, 1).otherwise(0)).alias("_claim_rows"),
    F.round(F.sum(TARGET), 2).alias("_target_sum"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in MOTOR_FEATURES],
).first()

n_rows = s["_rows"]
print(f"Motor modeling dataset: {n_rows:,} rows | {len(MOTOR_FEATURES)} features | target = {TARGET} (nulls->0)")
print(f"Rows with target > 0: {s['_claim_rows']:,} ({100.0*s['_claim_rows']/n_rows:.1f}%) | total target sum: {s['_target_sum']:,.0f}")
print(f"Zero-target rows: {n_rows - s['_claim_rows']:,} ({100.0*(n_rows - s['_claim_rows'])/n_rows:.1f}%)")

feat_nulls = {c: s[c] for c in MOTOR_FEATURES if s[c] and s[c] > 0}
print("\nFeatures with missing values (left as-is — XGBoost handles them natively):")
for c, k in sorted(feat_nulls.items(), key=lambda x: -x[1]):
    print(f"  {c:<22} {100.0*k/n_rows:6.2f}%")

from collections import Counter
_dtypes = {c: t for c, t in motor_model.dtypes if c in MOTOR_FEATURES}
print(f"\nFeature dtypes: {dict(Counter(_dtypes.values()))}")

display(motor_model.limit(10))